# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manalchaudharyy/FlyrankAI-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.***My rule (plain words):** flag pages that get real search visibility (impressions) but are
underperforming CTR relative to what pages at their SAME position tier typically achieve.
That gap — expected CTR for that position vs actual CTR — times how much traffic (impressions)
is riding on the page, is the opportunity size. This is the session's "CTR-fix" logic,
applied to my Lane 2 slice.

**Reason code (one, fixed):** `CTR_UNDERPERFORM_VS_POSITION_TIER`
**Action label:** `review_metadata_ctr_fix`

**Signal 1 — CTR vs. position tier** (this IS the signal behind the session's CTR-fix flag):
does CTR really fall as position gets worse? If not, my whole rule's assumption is broken.

**Signal 2 — Impression volume** (this is the "quick-win" signal): does traffic volume
concentrate in a way that makes prioritizing by impressions worthwhile, or is opportunity
scattered evenly regardless of volume?

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, os
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MONTH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"

print("Connected. Using mid-panel month=2026-03 (never the sealed final month).")

Connected. Using mid-panel month=2026-03 (never the sealed final month).


In [16]:
signal1 = con.sql(f"""
    WITH agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{FACT_MONTH}')
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT
        CASE
            WHEN avg_position <= 3 THEN '1_top_3'
            WHEN avg_position <= 10 THEN '2_top_10'
            WHEN avg_position <= 20 THEN '3_top_20'
            ELSE '4_beyond_20'
        END AS position_tier,
        COUNT(*) AS n,
        ROUND(AVG(clicks / NULLIF(impressions, 0)), 4) AS avg_ctr
    FROM agg
    GROUP BY 1
    ORDER BY 1
""").df()

print(signal1)
print("\nVerdict:", "CONFIRMED - CTR drops as position tier worsens (session's CTR-fix logic holds)"
      if signal1["avg_ctr"].is_monotonic_decreasing else "MIXED/OPPOSITE - check the table, pattern isn't clean monotonic decline")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier      n  avg_ctr
0       1_top_3   9687   0.0037
1      2_top_10  52218   0.0033
2      3_top_20  24294   0.0024
3   4_beyond_20  29915   0.0013

Verdict: CONFIRMED - CTR drops as position tier worsens (session's CTR-fix logic holds)


In [17]:
signal2 = con.sql(f"""
    WITH agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks
        FROM read_parquet('{FACT_MONTH}')
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 1
    ),
    quartiled AS (
        SELECT
            *,
            NTILE(4) OVER (ORDER BY impressions) AS impression_quartile
        FROM agg
    )
    SELECT
        impression_quartile,
        COUNT(*) AS n,
        ROUND(AVG(impressions), 1) AS avg_impressions,
        ROUND(AVG(clicks / NULLIF(impressions, 0)), 4) AS avg_ctr
    FROM quartiled
    GROUP BY impression_quartile
    ORDER BY impression_quartile
""").df()

print(signal2)
print("\nVerdict: CONFIRMED - if avg_ctr in the top impression quartile (quartile 4) is still")
print("well below 1.0, there's real headroom to capture -> volume-weighting the score is")
print("justified (quick-win logic holds).")

   impression_quartile      n  avg_impressions  avg_ctr
0                    1  44185              6.0   0.0102
1                    2  44185             75.7   0.0028
2                    3  44184            477.8   0.0024
3                    4  44184           5792.5   0.0030

Verdict: CONFIRMED - if avg_ctr in the top impression quartile (quartile 4) is still
well below 1.0, there's real headroom to capture -> volume-weighting the score is
justified (quick-win logic holds).


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue = con.sql(f"""
    WITH agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS actual_ctr
        FROM read_parquet('{FACT_MONTH}')
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN avg_position <= 3 THEN 0.25
                WHEN avg_position <= 10 THEN 0.12
                WHEN avg_position <= 20 THEN 0.05
                ELSE 0.02
            END AS expected_ctr_for_tier
        FROM agg
    )
    SELECT
        content_hash_id,
        impressions,
        clicks,
        ROUND(avg_position, 1) AS avg_position,
        ROUND(actual_ctr, 4) AS actual_ctr,
        expected_ctr_for_tier,
        ROUND((expected_ctr_for_tier - actual_ctr) * impressions, 2) AS opportunity_score,
        'CTR_UNDERPERFORM_VS_POSITION_TIER' AS reason_code,
        'review_metadata_ctr_fix' AS action_label
    FROM tiered
    WHERE expected_ctr_for_tier > actual_ctr
    ORDER BY opportunity_score DESC
""").df()

print(queue.shape)
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(115973, 9)


,content_hash_id,impressions,clicks,avg_position,actual_ctr,expected_ctr_for_tier,opportunity_score,reason_code,action_label
0,content_eadb33b5df496f4a,617124.0,5668.0,2.4,0.0092,0.25,148613.00,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
1,content_ec2e0346994fb5a5,245276.0,1480.0,2.9,0.0060,0.25,59839.00,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
2,content_0e03de7680314cd5,221310.0,720.0,2.7,0.0033,0.25,54607.50,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
3,content_8d7d99f109e19aa2,203497.0,289.0,2.6,0.0014,0.25,50585.25,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
4,content_4ffe18112a5642e3,186983.0,586.0,2.3,0.0031,0.25,46159.75,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
5,content_987d251ee617d9c6,152806.0,940.0,2.8,0.0062,0.25,37261.50,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
6,content_545bb6cc7081ded3,122905.0,287.0,2.6,0.0023,0.25,30439.25,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
7,content_85703b835ab9e744,120868.0,1091.0,2.7,0.0090,0.25,29126.00,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
8,content_44f34c0a90047651,212404.0,24.0,7.3,0.0001,0.12,25464.48,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
9,content_7172a7fad43f0998,205867.0,862.0,3.4,0.0042,0.12,23842.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix


The bottom 10 rows (weakest opportunity scores) all share very low impressions (52-102 total)
and position ~20-22 -- barely past our >=50 impression cutoff. At this volume, actual_ctr swings
wildly on just 1-2 clicks (e.g. 1 click out of 52 impressions = 1.9% CTR by chance alone), so
their opportunity_score is mostly noise, not a real signal. These would be the first rows I'd
drop if I raised the impression floor from 50 to something higher like 200.

In [19]:
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written:", len(queue), "rows to work/outputs/baseline_action_score.csv")

Written: 115973 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)
top10

,content_hash_id,impressions,clicks,avg_position,actual_ctr,expected_ctr_for_tier,opportunity_score,reason_code,action_label
0,content_eadb33b5df496f4a,617124.0,5668.0,2.4,0.0092,0.25,148613.00,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
1,content_ec2e0346994fb5a5,245276.0,1480.0,2.9,0.0060,0.25,59839.00,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
2,content_0e03de7680314cd5,221310.0,720.0,2.7,0.0033,0.25,54607.50,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
3,content_8d7d99f109e19aa2,203497.0,289.0,2.6,0.0014,0.25,50585.25,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
4,content_4ffe18112a5642e3,186983.0,586.0,2.3,0.0031,0.25,46159.75,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
5,content_987d251ee617d9c6,152806.0,940.0,2.8,0.0062,0.25,37261.50,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
6,content_545bb6cc7081ded3,122905.0,287.0,2.6,0.0023,0.25,30439.25,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
7,content_85703b835ab9e744,120868.0,1091.0,2.7,0.0090,0.25,29126.00,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
8,content_44f34c0a90047651,212404.0,24.0,7.3,0.0001,0.12,25464.48,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
9,content_7172a7fad43f0998,205867.0,862.0,3.4,0.0042,0.12,23842.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check: confirm no future-window or label-derived columns used
print("Columns used in score:", ["gsc_impressions", "gsc_clicks", "gsc_avg_position"])
print("All columns are aggregated ONLY from month=2026-03, no future window, no label,")
print("no product flags (health_score, priority_score) were available/used.")

# Weak picks: look at rows with very low impressions relative to top of queue
weak = queue.tail(10)
weak

Columns used in score: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
All columns are aggregated ONLY from month=2026-03, no future window, no label,
no product flags (health_score, priority_score) were available/used.


,content_hash_id,impressions,clicks,avg_position,actual_ctr,expected_ctr_for_tier,opportunity_score,reason_code,action_label
115963,content_b7fa7e01be12494b,52.0,1.0,39.2,0.0192,0.02,0.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115964,content_0f589ea066727a34,52.0,1.0,20.3,0.0192,0.02,0.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115965,content_77692e9fb189c0fd,52.0,1.0,42.1,0.0192,0.02,0.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115966,content_79cf576e8c2aa93d,52.0,1.0,53.8,0.0192,0.02,0.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115967,content_fe6a34fdf615ce06,102.0,2.0,21.6,0.0196,0.02,0.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115968,content_6664d740a88a9e45,52.0,1.0,21.2,0.0192,0.02,0.04,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115969,content_0d5f604ed84bbe79,51.0,1.0,30.5,0.0196,0.02,0.02,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115970,content_0e65186118fab81d,51.0,1.0,31.9,0.0196,0.02,0.02,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115971,content_5c64ccf649182d99,51.0,1.0,49.2,0.0196,0.02,0.02,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix
115972,content_49b7edc3e66ed24a,51.0,1.0,31.2,0.0196,0.02,0.02,CTR_UNDERPERFORM_VS_POSITION_TIER,review_metadata_ctr_fix


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.